In [ ]:
pip install -U python-jobspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 MB 15.7 MB/s eta 0:00:00
  Attempting uninstall: regex
    Found existing installation: regex 2025.11.3
    Uninstalling regex-2025.11.3:
      Successfully uninstalled regex-2025.11.3
  Attempting uninstall: NUMPY
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
music21 9.9.1 requires numpy>=1.26.4, bu

In [ ]:
pip install numpy==2.4.1

  Using cached numpy-2.4.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.4 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.3
    Uninstalling numpy-1.26.3:
      Successfully uninstalled numpy-1.26.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-jobspy 1.1.82 requires NUMPY==1.26.3, but you have numpy 2.4.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.1 which is incompatible.


In [ ]:
!pip install langchain langchain-google-genai

In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (
    model = "gemini-2.5-flash", google_api_key = GEMINI_API_KEY
)

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from typing import Optional, List

class JobsInfo(BaseModel):
    site: Optional[List[str]] = Field(description="The job site to search for jobs on")
    search_term: str = Field(description="The post for which to search for jobs")
    google_search_term: str = Field(description="A string that contains post, location and time, used to search on Google Jobs")
    location: Optional[str] = Field(description="The location to search for jobs in")
    results_wanted: Optional[int] = Field(description="The number of results to return")
    hours_old: Optional[int] = Field(description="The number of hours to search for jobs in the past")
    country_indeed: Optional[str] = Field(description="The country to search for jobs in")

agent = create_agent(
    model=llm,
    response_format=JobsInfo  # Auto-selects ProviderStrategy
)

site=['naukri', 'indeed'] search_term='web developer' google_search_term='web developer jobs in Bangalore, India posted in the last 24 hours' location='Bangalore' results_wanted=1 hours_old=24 country_indeed='India'


In [ ]:
prompt = input("Enter gemini prompt: ")

Enter gemini prompt: I want jobs for psychologist in the last 5 days. I live in bengaluru, india. I am looking for jobs on Naukri and Linkedin.


In [ ]:
result = agent.invoke({
    "messages": [{"role": "user", "content": prompt}]
})

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

site=['Naukri', 'Linkedin'] search_term='psychologist' google_search_term='psychologist in bengaluru, india posted in the last 5 days' location='bengaluru, india' results_wanted=None hours_old=120 country_indeed='india'


In [ ]:
if result["structured_response"].site == None:
  site_name = ["indeed", "linkedin", "zip_recruiter", "google", "glassdoor", "bayt", "naukri", "bdjobs"]
else:
  site_name = result["structured_response"].site

if result["structured_response"].search_term == None:
  search_term = "engineer"
else:
  search_term = result["structured_response"].search_term

if result["structured_response"].google_search_term == None:
  google_search_term = "engineer jobs in San Francisco, since yesterday"
else:
  google_search_term = result["structured_response"].google_search_term

if result["structured_response"].location == None:
  location = "San Francisco, CA"
else:
  location = result["structured_response"].location

if result["structured_response"].hours_old == None:
  hours_old = 72
else:
  hours_old = result["structured_response"].hours_old

if result["structured_response"].results_wanted == None:
  results_wanted = 20
else:
  results_wanted = result["structured_response"].results_wanted

if result["structured_response"].country_indeed == None:
  country_indeed = "USA"
else:
  country_indeed = result["structured_response"].country_indeed

In [ ]:
import csv
from jobspy import scrape_jobs

jobs = scrape_jobs(
    site_name=site_name,
    search_term=search_term,
    google_search_term=google_search_term,
    location=location,
    results_wanted=results_wanted,
    hours_old=hours_old,
    country_indeed=country_indeed,

    # linkedin_fetch_description=True # gets more info such as description, direct job url (slower)
    # proxies=["208.195.175.46:65095", "208.195.175.45:65095", "localhost"],
)
print(f"Found {len(jobs)} jobs")
print(jobs.head())
jobs.to_csv("jobs.csv", quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False) # to_excel

Found 40 jobs
              id      site                                        job_url  \
0  li-4367093545  linkedin  https://www.linkedin.com/jobs/view/4367093545   
1  li-4367225062  linkedin  https://www.linkedin.com/jobs/view/4367225062   
2  li-4367240363  linkedin  https://www.linkedin.com/jobs/view/4367240363   
3  li-4363398459  linkedin  https://www.linkedin.com/jobs/view/4363398459   
4  li-4366937962  linkedin  https://www.linkedin.com/jobs/view/4366937962   

  job_url_direct                                              title  \
0           None                             Occupational Therapist   
1           None                Academic Counselor Work From Office   
2           None                         Pediatric Speech Therapist   
3           None                            Head Academic Counselor   
4           None  Associate Child Psychologist (Trainee) Interns...   

                           company                           location  \
0  Orange Child Develop